# Fig 5C — TF motif accessibility per cell state
Faithful reproduction of **08_PlotTFActivity Part A**. OLS coefficient of per-peak Log2FC (each state vs the rest) on the grouped HOCOMOCO motif matrix. Rows = ALL TFs significant (FDR≤0.1) in ≥1 state, ordered by the state where they peak (then strongest-first within a state); columns = states in NE→Differentiated order. `*` = FDR≤0.1; symmetric 3-stop blue-white-red scale with breaks ±max|coef|.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
# --- 08_PlotTFActivity Part A (cells 3-4), replicated ------------------------
coef = load_matrix('StateTFActivity_coef.csv')
fdr  = load_matrix('StateTFActivity_FDR.csv')
cols = [s for s in STATE_ORDER if s in coef.columns]
coef, fdr = coef[cols], fdr[cols]
sigTF = coef.index[(fdr <= FDR_THRESHOLD).sum(axis=1) >= 1]        # sig in >=1 state (no cap)
mat = coef.loc[sigTF]; star = (fdr.loc[sigTF] <= FDR_THRESHOLD).values
# order rows by WHERE they peak (max.col), then strongest first within that state
peak    = mat.values.argmax(axis=1)
peakval = mat.values[np.arange(len(mat)), peak]
order   = np.lexsort((-peakval, peak))
mat = mat.iloc[order]; star = star[order]
print(len(sigTF), 'sig TFs |  states:', ' -> '.join(cols))
lim = np.abs(mat.values).max()
fig, ax = plt.subplots(figsize=(3.2, max(4.0, 0.16*len(mat))), layout='constrained')
im = ax.imshow(mat.values, aspect='auto', cmap=ACTIVITY_CMAP, vmin=-lim, vmax=lim)
ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels([state_label(c) for c in mat.columns], rotation=90)
ax.set_yticks(range(len(mat))); ax.set_yticklabels(tf_labels(mat.index))
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if star[i, j]: ax.text(j, i, '*', ha='center', va='center', fontsize=8, color='black')
ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
# horizontal colorbar at the bottom (fits the tall heatmap better than a side bar)
cb = fig.colorbar(im, ax=ax, location='bottom', fraction=0.03, pad=0.05, aspect=35)
cb.set_label('TF motif accessibility (coef)'); cb.outline.set_linewidth(0.5)
ax.set_title('TF motif accessibility per cell state\n(NE \u2192 Differentiated;  * FDR\u22640.1)', fontsize=8)
savepanel(fig, 'Fig5C_StateTFActivity_heatmap')
